# 05 · pandas 表格資料

用 pandas 把列與欄組成的表格資料（tabular data）載入、檢視、篩選與聚合，建立資料分析的基本工作流。

## 學習目標
- 能用多種方式建立資料框（DataFrame），並理解列索引（index）與欄（columns）的結構。
- 能讀取常見檔案格式（CSV、Parquet）並快速檢視資料的形狀與型別。
- 能依條件選取欄位與過濾列，組合出想要的子表。
- 能用次數統計（value_counts）與分組聚合（groupby aggregation）做摘要分析。
- 能辨識並處理缺失值（NaN, Not a Number）。
- 能在 pandas 與 NumPy 陣列之間互相轉換。

## 建立 DataFrame 與基本結構

資料框（DataFrame）是「有標籤的二維表格」：每一列（row）有列索引（index），每一欄（column）有欄名。
單獨一欄是一個序列（Series），DataFrame 可看成多個共用同一組 index 的 Series 並排。

為什麼學：列與欄各自有名字，後續所有選取、過濾、聚合都靠這些標籤定位，這是整個 pandas 的基礎。

最常見的建立方式是由字典（dict）建立，鍵（key）變欄名，值（list）變整欄資料。

In [ ]:
# 概念：由字典 dict 建立 DataFrame，每個 key 變成一欄
import pandas as pd

# 造一組假的學生資料當示範用
students = {
    'name': ['小明', '小華', '小美', '阿強'],   # 姓名欄
    'score': [85, 72, 91, 68],                 # 分數欄
    'class': ['A', 'B', 'A', 'B'],             # 班級欄
}
df = pd.DataFrame(students)

print(df)
print('列索引 index：', list(df.index))   # 預設是 0,1,2,... 的整數索引
print('欄名 columns：', list(df.columns))
print('單獨取一欄是 Series，型別為：', type(df['score']))

### 由巢狀清單建立

若資料是「一列一筆」的巢狀清單（list of lists），可直接傳入並另外指定欄名。
兩種方式結果相同，選哪種看資料來源較順手。

In [ ]:
# 概念：由巢狀清單 list 建立，每個子 list 是一列，columns 指定欄名
rows = [
    ['小明', 85, 'A'],
    ['小華', 72, 'B'],
    ['小美', 91, 'A'],
]
df2 = pd.DataFrame(rows, columns=['name', 'score', 'class'])

print(df2)
# 技巧：也能自訂 index 取代預設整數索引，讓列也有名字
df3 = pd.DataFrame(rows, columns=['name', 'score', 'class'], index=['s1', 's2', 's3'])
print(df3)

## 載入資料檔

實務資料多半來自檔案。先掌握「寫出再讀回」的最小循環：`to_csv` / `to_parquet` 寫出，`read_csv` / `read_parquet` 讀回。

兩種格式差異：

| 格式 | 性質 | 特點 |
|---|---|---|
| CSV | 純文字 | 人類可讀、通用，但無型別資訊、檔案較大 |
| Parquet | 欄式二進位 | 保留型別、壓縮率高、讀取快，但非純文字 |

本範例用程式自造的暫存檔示範，全程不依賴任何外部檔案。

In [ ]:
# 概念：寫出再讀回的最小循環（先 to_csv / to_parquet，再 read_csv / read_parquet）
import os
import tempfile

df = pd.DataFrame({
    'name': ['小明', '小華', '小美'],
    'score': [85, 72, 91],
})

# 用系統暫存資料夾，避免在專案留下檔案
tmp_dir = tempfile.gettempdir()
csv_path = os.path.join(tmp_dir, 'demo_students.csv')

df.to_csv(csv_path, index=False)        # 注意：index=False 才不會把列索引也寫進檔案
back_csv = pd.read_csv(csv_path)        # 讀回成新的 DataFrame
print('CSV 讀回：')
print(back_csv)

# 技巧：Parquet 需要額外引擎（pyarrow），環境若沒裝就略過，不影響其他單元
try:
    pq_path = os.path.join(tmp_dir, 'demo_students.parquet')
    df.to_parquet(pq_path)
    back_pq = pd.read_parquet(pq_path)
    print('Parquet 讀回（型別被保留）：')
    print(back_pq.dtypes)
except Exception as e:
    print('Parquet 未啟用（缺 pyarrow）：', e)

## 檢視與摘要資料

拿到一份新資料的標準第一步：先看大小、前幾列、欄名與每欄型別（dtype），再決定下一步。

| 屬性 / 方法 | 回傳 |
|---|---|
| `.shape` | （列數, 欄數）的元組 |
| `.head(n)` | 前 n 列（預設 5） |
| `.columns` | 欄名 |
| `.dtypes` | 每欄的資料型別（dtype） |
| `.to_numpy()` | 把整張表降為 NumPy 陣列 |

In [ ]:
# 概念：檢視一份新資料的標準四件事（shape / head / columns / dtypes）
import numpy as np

# 造一份多欄、較多列的假資料
df = pd.DataFrame({
    'city': ['台北', '台中', '高雄', '台北', '台南', '台中'],
    'population': [264, 281, 277, 264, 187, 281],   # 萬人
    'area_km2': [271.8, 2214.9, 2951.9, 271.8, 2191.7, 2214.9],
})

print('形狀（列, 欄）：', df.shape)
print('前 3 列：')
print(df.head(3))
print('欄名：', list(df.columns))
print('每欄型別：')
print(df.dtypes)
print('轉成 NumPy 陣列（會混合型別變 object）：')
print(df.to_numpy())

## 欄位選取與條件過濾

分析常從「只看我關心的欄與符合條件的列」開始。

- 選單欄：`df['欄名']`（得到 Series）。
- 選多欄：`df[['欄A', '欄B']]`（傳入欄名 list，得到 DataFrame）。
- 過濾列：先用比較運算造出布林遮罩（boolean mask），再把遮罩放回 `df[...]` 只留下 True 的列。

形狀：`df[df['欄名'] > 門檻]`

In [ ]:
# 概念：布林遮罩 boolean mask 過濾列，再選取想要的欄
df = pd.DataFrame({
    'name': ['小明', '小華', '小美', '阿強', '小芳'],
    'score': [85, 72, 91, 68, 88],
    'class': ['A', 'B', 'A', 'B', 'A'],
})

mask = df['score'] > 80          # 對每一列做比較，得到一整欄 True/False
print('布林遮罩：')
print(mask.tolist())

high = df[mask]                  # 只留下遮罩為 True 的列
print('分數大於 80 的列：')
print(high)

# 技巧：過濾與選欄可一氣呵成；多條件用 & | 並各自加括號
result = df[(df['score'] > 80) & (df['class'] == 'A')][['name', 'score']]
print('A 班且分數大於 80，只看姓名與分數：')
print(result)

## 次數統計與分組聚合

把細項資料壓縮成有意義的摘要，是表格分析的核心。

- `value_counts()`：數單欄各類別出現幾次。
- `groupby('欄')`：依某欄的類別分組，再對各組套用聚合函式（aggregation）如 `mean`、`sum`、`count`。

形狀：`df.groupby('分組欄')['數值欄'].mean()`

In [ ]:
# 概念：value_counts 數次數，groupby 分組後聚合 aggregation
df = pd.DataFrame({
    'name': ['小明', '小華', '小美', '阿強', '小芳', '阿明'],
    'score': [85, 72, 91, 68, 88, 75],
    'class': ['A', 'B', 'A', 'B', 'A', 'B'],
})

print('各班級出現次數：')
print(df['class'].value_counts())

print('各班級平均分數：')
print(df.groupby('class')['score'].mean())

# 技巧：用 agg 一次求多個統計量，欄名清楚
summary = df.groupby('class')['score'].agg(['mean', 'count', 'max'])
print('各班級摘要（平均、人數、最高分）：')
print(summary)

## 缺失值處理

真實資料常有空缺，pandas 用缺失值（NaN, Not a Number）表示。

1. 偵測：`isna()` 回傳布林遮罩，標出哪裡是缺值。
2. 丟棄：`dropna()` 直接移除含缺值的列。
3. 填補：`fillna(值)` 用合理的值（如中位數、0）補上。

為什麼學：缺值會汙染統計（如平均），必須先決定丟棄或填補。

In [ ]:
# 概念：缺失值 NaN 的偵測 isna、丟棄 dropna、填補 fillna
# 注意：用 np.nan 在數值欄刻意製造缺值
df = pd.DataFrame({
    'name': ['小明', '小華', '小美', '阿強'],
    'score': [85, np.nan, 91, np.nan],
})

print('偵測缺值（每格是否為 NaN）：')
print(df['score'].isna().tolist())

print('丟棄含缺值的列：')
print(df.dropna())

# 技巧：填補前先算未缺值的中位數，避免被 NaN 影響（中位數本身會自動略過 NaN）
median_score = df['score'].median()
filled = df.fillna({'score': median_score})
print('用中位數', median_score, '填補後：')
print(filled)

## DataFrame 與 NumPy 互轉

當需要對整批數值做向量化運算時，常把表格降為純 NumPy 陣列計算，算完再加回欄名變回表格。

- 表格轉陣列：`df.to_numpy()` 或 `df[['欄...']].to_numpy()`。
- 陣列轉表格：`pd.DataFrame(陣列, columns=[...])`。

取捨：純陣列計算快但失去欄名與型別資訊；加回欄名才好閱讀與後續分析。

In [ ]:
# 概念：DataFrame 轉 NumPy 做向量化運算，再包回帶欄名的 DataFrame
df = pd.DataFrame({
    'math': [80, 65, 90],
    'english': [70, 88, 60],
})

arr = df.to_numpy()              # 取出純數值陣列（無欄名）
print('降為陣列：')
print(arr)

# 向量化：整批同時加 5 分（廣播 broadcasting），不需迴圈
adjusted = arr + 5

# 把運算結果包回 DataFrame，重新貼上欄名
result = pd.DataFrame(adjusted, columns=df.columns)
print('整體加 5 分後重新變回表格：')
print(result)
# 技巧：也可直接 df + 5，pandas 會保留欄名；此處示範「降階再升階」的工作流
print('每位學生兩科總分（用陣列沿欄方向相加）：', arr.sum(axis=1))

## 練習

以下三題由簡到難，皆為複合型但簡短。資料自己用字典 / list / numpy 造（仿真即可），不引用任何外部檔案。只需在 `# TODO` 處補上程式。

In [ ]:
# TODO 1 ·（簡單）建築基本資料表（整合：建立 DataFrame + 欄位選取與條件過濾）
#   情境：手上有幾棟建築的樓高 height（公尺）與樓層數 floors，想挑出高樓。
#   要求：
#     1. 用字典 dict 自造一個小 DataFrame，欄位為 building_id、height、floors，
#        資料用 list 直接寫幾筆仿真數字即可。
#     2. 用布林遮罩 boolean mask 過濾出 height 大於 30 的列，
#        並只選取 building_id 與 height 兩欄。
#   驗收：應該看到一張只剩高樓、且只有兩欄的子表。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）用途分類的容積摘要
#   （整合：建立 DataFrame + 缺失值處理 + value_counts + groupby 聚合）
#   情境：一批建築有用途標籤 usage（住宅 / 商業 / 工業）與樓地板面積 gfa（gross floor area），
#        部分 gfa 缺漏，想看各用途的規模摘要。
#   要求：
#     1. 自造含 usage 欄與 gfa 欄的 DataFrame，並刻意讓幾筆 gfa 為缺失值（用 np.nan）。
#     2. 用 isna 偵測缺值後，以中位數或 0 等合理策略用 fillna 填補。
#     3. 用 value_counts 數各 usage 出現的次數。
#     4. 用 groupby 依 usage 聚合，求各類的平均 gfa 與棟數（count）。
#   驗收：應該看到一張以 usage 為列、含平均 gfa 與棟數的摘要表，且資料不再有缺值。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）街廓網格的容積率前後比較
#   （整合：建立 DataFrame + DataFrame 與 NumPy 互轉 + 條件過濾 + groupby 聚合）
#   情境：每棟建築屬於某個街廓網格 grid，已知占地面積 footprint 與樓地板面積 gfa；
#        要評估一個「高度乘數」政策情境下，各網格的容積率 FAR（floor area ratio）如何變化。
#   要求：
#     1. 自造含 grid、footprint、gfa 三欄的 DataFrame。
#     2. 用 .to_numpy() 取出 gfa 與 footprint 兩數值欄，向量化算現況 FAR = gfa / footprint；
#        再對 gfa 乘上高度乘數（如 1.2）算政策後 FAR。
#     3. 把現況 FAR 與政策後 FAR 兩結果包回帶欄名的 DataFrame（連同 grid 欄）。
#     4. 過濾出政策後 FAR 超過上限（如 3.0）的列，並用 groupby 依 grid 聚合，
#        求各網格政策後 FAR 的平均與超標棟數。
#   驗收：應該看到一張各網格的聚合表，能看出哪些網格政策後平均容積率偏高、有幾棟超標。
# TODO: 在這裡完成


## 小結

- DataFrame 是有列索引（index）與欄名的二維表格，單欄是 Series；可由字典或巢狀清單建立。
- 用 `to_csv` / `read_csv`、`to_parquet` / `read_parquet` 完成「寫出再讀回」的檔案循環；CSV 為純文字、Parquet 為欄式二進位。
- 拿到新資料先看 `.shape`、`.head()`、`.columns`、`.dtypes`。
- 用布林遮罩過濾列、用欄名 list 選多欄，組出想要的子表。
- `value_counts` 數次數，`groupby` 分組後用聚合函式做摘要。
- 缺失值（NaN）用 `isna` 偵測、`dropna` 丟棄、`fillna` 填補。
- 用 `.to_numpy()` 把表格降為陣列做向量化運算，再包回帶欄名的 DataFrame。